# Assignment: Barcelona

## Prep

### Imports, shared definitions, datasets

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily
import pointpats
import numpy as np
import seaborn as sns
from libpysal import graph
import esda

In [ ]:
def listings(city_path, quarter_end_dates):
    """load listings for a city_path for a series of dates (representing quarter ends) and combines into one DataFrame.

    Adds a new `quarter_end_date` column so that data for each quarter can still be pulled out later.
    """
    listings_url_base = f"https://data.insideairbnb.com/{city_path}"
    dfs = []
    for quarter_end_date in quarter_end_dates:
        listings_url = f"{listings_url_base}/{quarter_end_date}/data/listings.csv.gz"
        df = pd.read_csv(listings_url, compression='gzip')
        df['quarter_end_date'] = quarter_end_date
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [ ]:
listings_df = listings("spain/catalonia/barcelona", ["2024-12-12","2025-03-05","2025-06-12","2025-09-14"])
listings_df

In [ ]:
def create_gdf_from_latlon(df):
    """creates a new GDF from a Dataframe containing `latitude` and `longitude` columns."""
    geometry = gpd.points_from_xy(df['longitude'], df['latitude'], crs="EPSG:4326")
    return gpd.GeoDataFrame(df, geometry=geometry)

In [ ]:
listings_gdf = create_gdf_from_latlon(listings_df)
listings_gdf

In [ ]:
def price_only(gdf):
    """Takes a GDF with a `price` column formatted as a $100.00 string and turns it into a float `price` column.
    The `price` and geometry columns are returned in a new GDF.
    """
    price_gdf = gdf.copy(deep=True)
    price_gdf["price"] = (
        gdf["price"]
          .str.replace(r"[\$,]", "", regex=True)
          .astype(float)
    )
    price_gdf = price_gdf[["price", gdf.geometry.name]]
    price_gdf = price_gdf.dropna()
    return price_gdf
    
listings_price_gdf = price_only(listings_gdf)
listings_price_gdf.head()

In [ ]:
listings_price_gdf.explore("price", tiles="CartoDB Positron", scheme='percentiles', prefer_canvas=True)

In [ ]:
import h3

def point_to_h3_fn(h3_res):
    """returns a new function which will convert a POINT geometry into an H3 id"""
    def f(row):
        return h3.latlng_to_cell(row.geometry.y, row.geometry.x, h3_res)
    return f

def add_grid_cells(gdf, h3_res):
    """adds a new `h3_id` column to a GDF which is assumed to have a POINT geometry"""
    expected_crs = "EPSG:4326"
    assert gdf.crs == expected_crs, f"needed a CRS of {expected_crs}, but this was {gdf.crs}"
    gdf["h3_id"] = gdf.apply(point_to_h3_fn(h3_res), axis=1)
    
    

In [ ]:
add_grid_cells(listings_price_gdf, h3_res=10)
listings_price_gdf.head()

In [ ]:
# show price distribution

In [ ]:
sns.displot(listings_price_gdf["price"])

In [ ]:
# the distribution is very skewed so we'll use median rather than mean as a summary of each cell

In [ ]:
from shapely.geometry import Polygon

def h3_to_polygon(h3_id):
    """takes an H3 and returns a boundary as a Shapely Polygon"""
    boundary = h3.cell_to_boundary(h3_id)
    lng_lat = [(lng, lat) for lat, lng in boundary]
    return Polygon(lng_lat)
    
def summary_price(gdf):
    """takes a GDF with a 'price' and 'h3_id' column and returns a new GDF with the median price per 'h3_id',
    and a geometry column which is the boundary of the cell as a Polygon"""
    median_price_group = gdf.groupby(["h3_id"])["price"].median()
    summary_gdf = gpd.GeoDataFrame(
        median_price_group,
        geometry=[h3_to_polygon(h) for h in median_price_group.index],
        crs='EPSG:4326'
    )
    return summary_gdf


In [ ]:
listings_price_summary_gdf = summary_price(listings_price_gdf)
listings_price_summary_gdf.head()

In [ ]:
m = listings_price_summary_gdf.explore("price", tiles="CartoDB Positron", cmap="Blues", scheme="percentiles", prefer_canvas=True)
listings_price_gdf.explore("price", m=m, cmap="Reds", scheme="percentiles", prefer_canvas=True)
m

In [ ]:
def build_h3_contiguity(gdf):
    """This takes GDF with an `h3_id` and builds a normalised `libpysal.graph.base.Graph` and a filtered GDF.
    
    It will remove any H3 cells which are isolates.

    It returns the original GDF, but with the filtered cells removed, and the contiguity
    
    """
    # get the contiguity without any filtering
    contiguity = graph.Graph.build_contiguity(gdf, rook=False)
    
    # join back to original GDF and filter out any isolates
    cardinalities = contiguity.cardinalities
    cardinalities.name='cardinalities'
    joined = gdf.join(cardinalities)
    filtered = joined[joined['cardinalities'] != 0].drop('cardinalities', axis=1)
    
    # rebuild the contiguity from the filtered GDF
    filtered_contiguity = graph.Graph.build_contiguity(filtered, rook=False)
    
    return filtered, filtered_contiguity

In [ ]:
listings_price_summary_gdf, listings_price_summary_contiguity = build_h3_contiguity(listings_price_summary_gdf)

In [ ]:
listings_price_summary_gdf

In [ ]:
listings_price_summary_contiguity

In [ ]:
m = listings_price_summary_gdf.explore()
listings_price_summary_contiguity.explore(
    listings_price_summary_gdf, m=m, edge_kws=dict(style_kwds=dict(weight=1)), nodes=False, focal=h3_id
)

In [ ]:
sns.displot(listings_price_summary_contiguity.cardinalities, bins=6)

In [ ]:
def prepare_for_moran(gdf, contiguity):
    """prepares a GDF on `price` for analysis via Moran Global and Local.
    
    returns a new prepared GDF with a new `price_std` column and a transformed contiguity.

    note that it returns only the columns needed for Moran to avoid accidentally using the wrong columns.
    """
    gdf_std = gpd.GeoDataFrame(columns=['geometry'],crs=gdf.crs)
    gdf_std["price_std"] = (
        gdf["price"] - gdf["price"].mean()
    ) / gdf["price"].std()
    contiguity_r = contiguity.transform("r")
    gdf_std["price_std_lag"] = contiguity_r.lag(gdf_std["price_std"])
    return gdf_std, contiguity_r

In [ ]:
listings_price_summary_std_gdf, listings_price_summary_contiguity_r = prepare_for_moran(
    listings_price_summary_gdf, listings_price_summary_contiguity
)

In [ ]:
h3_id = "8a3944601037fff"

In [ ]:
listings_price_summary_std_gdf[listings_price_summary_std_gdf.index == h3_id]

In [ ]:
listings_price_summary_contiguity_r[h3_id]

In [ ]:
f, ax = plt.subplots(1, figsize=(6, 6))
sns.regplot(
    x="price_std",
    y="price_std_lag",
    data=listings_price_summary_std_gdf,
    marker=".",
    scatter_kws={"alpha": 0.2},
    line_kws=dict(color="lightcoral")
)
ax.set_aspect('equal')
plt.axvline(0, c="black", alpha=0.5)
plt.axhline(0, c="black", alpha=0.5)


In [ ]:
def moran_global(gdf, contiguity_r):
    moran = esda.Moran(gdf['price_std'], contiguity_r)
    print(f"I: {moran.I}, P_sim: {moran.p_sim}")

In [ ]:
moran_global(listings_price_summary_std_gdf, listings_price_summary_contiguity_r)

In [ ]:
def moran_local_clusters(gdf, contiguity_r):
    lisa = esda.Moran_Local(gdf['price_std'], contiguity_r)
    cluster_gdf = gpd.GeoDataFrame(columns=['geometry'],crs=gdf.crs)
    cluster_gdf['cluster'] = lisa.get_cluster_labels(crit_value=0.05)
    return cluster_gdf, lisa

In [ ]:
listings_price_summary_std_clusters_gdf, lisa = moran_local_clusters(
    listings_price_summary_std_gdf, listings_price_summary_contiguity_r
)

In [ ]:
lisa.explore(
    listings_price_summary_gdf,
    crit_value=0.05,
    prefer_canvas=True,
    tiles="CartoDB Positron",
)

In [ ]:
_ = lisa.plot_scatter()